# The STFT frequency-summary experiment

One hypothesis, one run, one pre-registered bar.

`STFTBranch` ends with `f.mean(dim=2)`, which averages the frequency axis
away. "Peak at bin 2, then bin 6" (FHSS hopping) and "peak at bin 4, then
bin 4" (tone jamming) come out identical. That is why FHSS recall falls off a
cliff as the jammer gains power -- 0.993 with no jammer, 0.533 at +5 dB,
**0.048 at +10 dB** -- even though the information is still there: FHSS sits in
10-48 kHz channels while barrage jamming spreads over 200 kHz - 1.2 MHz, so a
frequency-selective detector has 10-20 dB of processing gain available.

With `model.stft_freq_summary` on, the branch pools time only (keeping 200 kHz
bins instead of 400) and adds three per-frame features taken from the STFT
magnitude directly: frequency max, spectral flatness, and peak-frequency delta.

### The bar, fixed before the run

| | baseline | pass |
|---|---|---|
| FHSS recall at +10 dB JSR | 0.048 | **> 0.25** |
| JAMMING recall alone | 0.992 | must not collapse |
| jammer called FHSS | 0.017 | must not materially rise |

That third row is the trap. FHSS and JAMMING share a decision boundary, and
across three earlier fixes FHSS recall climbed 82.5 -> 89.7 -> 92.2 while
JAMMING fell 80.0 -> 73.3 -> 67.5 in the same runs. A win that comes from the
model relabelling jammers as FHSS is not a win, so the probe reports it.

### Why this one is worth the GPU time and the last two were not

`cumulant_features` and `if_features` both failed: neither moved its target
and both dropped FHSS recall below the 0.80 gate. Each bolted one scalar onto
the fused vector, where it had to compete with 192 learned dimensions. This
changes the architecture that discards the information in the first place.


## 1. GPU on

**Runtime -> Change runtime type -> T4 GPU -> Save**, before anything else.
Changing it later restarts the machine and wipes whatever you had loaded.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'CPU ONLY -- change the runtime type, then run this again')


## 2. Code

Colab clones from GitHub and cannot see your laptop, so the branch has to be
pushed. This notebook lives on `eileen-stft-experiment`, which branches from
the pushed tip of `eileen-omni-ui` and carries nothing else.


In [ ]:
%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b eileen-stft-experiment https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log --oneline -3
!pip install -q pyyaml h5py


## 3. Data

Three arrays that are not in git. Mount Drive and copy inside Google's network.

**Colab mounts My Drive only.** If your data sits under a Drive-for-Desktop
computer mirror (a top-level folder named after your machine, e.g. `My Laptop`),
it is *not* reachable from here -- the search cell below will not find it, and
you would need to copy it into My Drive first.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Find X.npy anywhere in My Drive, so you don't have to guess the path.
import subprocess
hits = subprocess.run(['find', '/content/drive/MyDrive', '-name', 'X.npy'],
                      capture_output=True, text=True).stdout.split()
for h in hits:
    print(h)
print('\n-> set DRIVE_DATA below to the FOLDER holding the copy you want')
if not hits:
    print('nothing found: the data is probably in a computer mirror, not My Drive')


In [ ]:
import shutil, pathlib

DRIVE_DATA = '/content/drive/MyDrive/sedicAI_NEXA/data/processed'   # EDIT ME

dest = pathlib.Path('/content/sedicAI_NEXA/data/processed')
dest.mkdir(parents=True, exist_ok=True)
for name in ('X.npy', 'y.npy', 'snr_labels.npy'):
    src = pathlib.Path(DRIVE_DATA) / name
    assert src.exists(), f'not found: {src} -- fix DRIVE_DATA above'
    shutil.copy(src, dest / name)
    print(f'{name:16s} {(dest / name).stat().st_size / 1e6:8.1f} MB')

import numpy as np
X = np.load(dest / 'X.npy', mmap_mode='r')
assert X.shape[1:] == (2, 512), f'unexpected shape {X.shape}'
print('X', X.shape, '-- expect (80400, 2, 512)')


## 4. Turn the flag on -- and prove it took

The flag is set in memory only. It is never written to `configs/default.yaml`,
because with it on the fused tensor changes shape and **none of the five
shipped checkpoints will load** -- leaving it enabled on disk would break the
console and the submission.

The assert below is the point of this cell. A flag that silently fails to apply
produces a run that looks fine and measures nothing, which is the most
expensive kind of mistake available here.


In [ ]:
import sys; sys.path.insert(0, '/content/sedicAI_NEXA')
from src.config import CFG, CLASSES
CFG.setdefault('model', {})['stft_freq_summary'] = True

from src.models.amc_cnn import AMC_CNN
probe = AMC_CNN(num_classes=len(CLASSES), input_len=CFG['signal']['window_len'])
fc1 = tuple(probe.fc1.weight.shape)
print('fc1', fc1, ' shipped architecture is (256, 192)')
assert fc1 != (256, 192), 'flag did NOT apply -- stop, do not train'
print('flag applied; this is a different architecture')
del probe


## 5. Train

Seed 2000 -- ensemble member 0, the seed the pinned baseline was measured on.

The full 30 epochs, even though best validation has landed around epoch 9 in
recent runs. Comparability against `jsr_baseline_1model.json` is worth more
than the ten minutes an early stop would save on a T4.

The checkpoint is written every time validation improves, so a disconnect
costs you the tail of the run, not the run. **Do not close the tab** -- Colab
kills idle sessions.


In [ ]:
import pathlib, torch
from src.train import load_data, stratified_split
from scripts.train_ensemble import train_one

SEED = 2000
X, y, snr_labels = load_data()
d = CFG['dataset']
tr, va, _ = stratified_split(y, snr_labels, d['val_frac'], d['test_frac'], d['seed'])
print(f'train {len(tr)}  val {len(va)}')

out  = pathlib.Path('/content/sedicAI_NEXA/results/experiment_stft_freq_summary.pt')
hist = pathlib.Path('/content/sedicAI_NEXA/results/experiment_stft_freq_summary_history.json')
out.parent.mkdir(parents=True, exist_ok=True)

model = train_one(X, y, snr_labels, tr, va, seed=SEED,
                  history_path=hist, ckpt_path=out)
torch.save(model.state_dict(), out)
print('saved', out)


## 6. Measure

`--stft-freq-summary` is required: the flag changed the fused width, so the
checkpoint only loads into a model built the same way.

`--n 600 --seed 0` matches the baseline exactly. Do not change either, or the
comparison stops meaning anything.


In [ ]:
!python scripts/probe_jsr.py --n 600 --seed 0 \
    --checkpoint results/experiment_stft_freq_summary.pt \
    --stft-freq-summary \
    --out results/jsr_freqsummary_1model.json


## 7. Did it pass?

Read against the bar set at the top, not against whatever looks encouraging.


In [ ]:
import json

base = {r['jsr_db']: r for r in json.load(
    open('docs/experiments/jsr_baseline_1model.json'))['rows']}
new  = {r['jsr_db']: r for r in json.load(
    open('results/jsr_freqsummary_1model.json'))['rows']}

print('   JSR      FHSS recall        jammer called FHSS')
print('              base       new         base         new')
for k in (None, 0, 5, 10, 15, 20):
    b, n = base[k], new[k]
    label = 'none' if k is None else '+%d dB' % k
    print('%6s  %10.3f %9.3f   %10.3f  %10.3f' % (
        label, b['fhss_recall'], n['fhss_recall'],
        b['jammer_called_fhss'], n['jammer_called_fhss']))

f10   = new[10]['fhss_recall']
alone = new[0]['jamming_recall_alone']
leak  = new[10]['jammer_called_fhss']
print()
print('FHSS @ +10 dB   %.3f   bar > 0.25, baseline 0.048   %s' % (
    f10, 'PASS' if f10 > 0.25 else 'FAIL'))
print('JAMMING alone   %.3f   baseline 0.995               %s' % (
    alone, 'ok' if alone > 0.95 else 'COLLAPSED'))
print('jammer -> FHSS  %.3f   baseline 0.017               %s' % (
    leak, 'ok' if leak < 0.10 else 'LEAKING -- the win is relabelling'))


## 8. Save to Drive before the tab closes

`/content` is deleted when the session ends. Save the probe JSON too, not just
the checkpoint -- it is the actual result, and it is 4 KB.


In [ ]:
import shutil, pathlib
DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/sedicAI_NEXA_experiments')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
for f in (out, hist, pathlib.Path('results/jsr_freqsummary_1model.json')):
    shutil.copy(f, DRIVE_OUT / f.name)
    print('saved to Drive:', f.name)


---

### If it passes

One member is not a decision. The effect being tested is roughly 5x a single
model's seed spread, which is why one member is enough to *detect* it -- but
confirming it means retraining all five with the flag on, re-running the full
scorecard, and checking the judged-class gate holds at 0.80. Budget for that
before switching the default.

### If it fails

That closes the frequency-selectivity hypothesis, and with the two expert-
feature branches already negative it closes the architectural avenue for FHSS
under jamming. The honest reading then is that the +10 dB cliff is a property
of a 512-sample window rather than of this model, and it belongs in the
limitations section alongside the 16QAM/64QAM ceiling -- measured, explained,
and not papered over.
